# 05 — Two-Stage Prediction Model: Stage Classification -> Metabolite Prediction

This notebook implements a two-stage architecture for predicting polyamine
metabolite abundance in the Yachida CRC cohort:

1. **Stage 1**: A LightGBM multiclass classifier predicts CRC disease-severity
   group probabilities from species abundances and clinical confounders.
   Out-of-fold (OOF) probability vectors are generated via 10-fold
   `StratifiedKFold` cross-validation to prevent data leakage. The label is
   the **grouped, corrected** 4-class severity scheme from `utils` --
   `Healthy` / `Adenoma` (MP, multiple polypoid adenomas) / `Early_CRC` (Stage_0 +
   Stage_I_II) / `Advanced_CRC` (Stage_III_IV) -- via `utils.STAGE_GROUP_MAP` /
   `utils.STAGE_GROUP_NUMERIC`. HS (history of colorectal surgery) is excluded from v2.
   The old pipeline put MP (a cancer-free adenoma group) in the same bucket as
   Stage_III_IV (regionally/distantly advanced cancer) -- see
   `utils.py`'s "STAGE ENCODING" section for the full rationale. We import
   the grouping from `utils` rather than re-deriving it locally so this
   notebook can never drift out of sync with that fix.

2. **Stage 2**: A Ridge regression model predicts polyamine abundance for
   each pre-specified polyamine target (NB03's `ml_results_v2.pkl`,
   `polyamine_targets`), using
   species features, Stage 1 OOF probabilities, and confounders as inputs.
   A "flat" baseline (species + confounders only, no Stage 1 probabilities)
   is compared via 10-fold CV Pearson R to quantify the added value of
   stage information.

## Two leakage bugs fixed relative to the old `exploratory/05_two_stage_model.ipynb`

The old notebook called `scaler.fit_transform(X)` on the **full** sample set
before `cross_val_predict(...)`, and computed each target metabolite's
median-imputation value over the **full** sample set before CV. Both leak
held-out-fold information into every fold's training statistics. Here,
scaling (and confounder imputation) happen **inside** a
`sklearn.pipeline.Pipeline` passed as a whole into `cross_val_predict(...)`,
so `cross_val_predict` clones and re-fits the imputer/scaler/model on each
fold's *training* rows only. Target imputation is dropped entirely rather
than made fold-safe: `transformed_mtb` (from NB01) already has no missing
values by construction (`utils.impute_half_min` runs before log-transform),
so any residual NaN after merging is a genuine row-alignment gap, not a
value to synthesize -- those rows are dropped outright, which cannot leak
anything (a discarded row carries no information into any fold).

**No disease-status proxy among the features.** The confounders are Age, BMI,
Brinkman.Index, Alcohol and Gender only. An earlier version also fed one-hot
`Tumor.Location.Category` into the Stage-1 classifier; "NoTumor" exactly separates
{Healthy, Adenoma} from {Early, Advanced} CRC, so it inflated Stage-1 accuracy.

**Scope note**: no downstream v2 notebook reads this notebook's outputs --
it is a self-contained leaf, same as the old pipeline's A9 stage.


In [ ]:
import re
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (registers IterativeImputer)
from sklearn.impute import IterativeImputer
from sklearn.metrics import classification_report, cohen_kappa_score
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')

sys.path.insert(0, '.')  # run this notebook from v2/
from utils import (
    RESULTS_DIR, POLYAMINE_KEGG,
    STAGE_GROUP_NUMERIC, STAGE_GROUP_ORDER,
    build_confounder_matrix,
)

CRC_DS = 'YACHIDA-CRC-2019'
PKL_PREP = RESULTS_DIR / 'intermediate' / 'preprocessed_data_reduced_v2.pkl'
PKL_ML   = RESULTS_DIR / 'intermediate' / 'ml_results_v2.pkl'

with open(PKL_PREP, 'rb') as f:
    _prep = pickle.load(f)
with open(PKL_ML, 'rb') as f:
    ml_res = pickle.load(f)

spc     = _prep['transformed_species'][CRC_DS].copy()
mtb     = _prep['transformed_mtb'][CRC_DS].copy()
meta    = _prep['harmonized_meta'][CRC_DS].copy()
pa_cols = _prep['polyamine_columns'][CRC_DS]

print(f"Samples: {len(meta)}, Species: {spc.shape[1] - 1}, Metabolites: {mtb.shape[1] - 1}")
print(f"Stage.Group distribution (utils.STAGE_GROUP_MAP grouping):")
print(meta['Stage.Group'].value_counts().reindex(STAGE_GROUP_ORDER))


## 1. Data Preparation

Confounders come from `utils.build_confounder_matrix()`: Age, BMI, Brinkman.Index, Alcohol
and Gender. Tumor location is deliberately excluded (it is a disease-status proxy). Any
imputation of the genuinely-missing numeric columns must happen **inside** each CV fold, not
on this matrix as a whole -- that's done below, separately for Stage 1 and Stage 2.

In [ ]:
conf_df = build_confounder_matrix(meta)
assert not any(c.startswith('TumorLoc_') for c in conf_df.columns), 'disease-status proxy in features'

spc_feat_cols  = [c for c in spc.columns if c != 'Sample']
conf_feat_cols = [c for c in conf_df.columns if c != 'Sample']

data_merged = meta[['Sample', 'Study.Group', 'Stage.Group']].merge(spc, on='Sample').merge(conf_df, on='Sample')
print(f"Feature matrix: {data_merged.shape}")

# Grouped 4-class severity label -- Stage.Group was already derived upstream
# (utils.harmonize_metadata, via utils.STAGE_GROUP_MAP) so mapping it to a
# numeric class here via utils.STAGE_GROUP_NUMERIC is the single source of
# truth end to end; nothing is re-derived locally.
data_merged['Stage_class'] = data_merged['Stage.Group'].map(STAGE_GROUP_NUMERIC)
data_merged = data_merged.dropna(subset=['Stage_class']).reset_index(drop=True)
data_merged['Stage_class'] = data_merged['Stage_class'].astype(int)

n_classes = len(STAGE_GROUP_ORDER)
print(f"Class distribution ({STAGE_GROUP_ORDER}):")
print(data_merged['Stage.Group'].value_counts().reindex(STAGE_GROUP_ORDER))

samples = data_merged['Sample'].values
y_class = data_merged['Stage_class'].values.astype(int)


## 2. Stage 1 -- LightGBM 4-class severity classifier (OOF)

Species values have no missing entries by construction (CLR-transformed in NB01). Confounder columns may have residual NaN. LightGBM handles missing feature values natively (it learns an optimal default split direction for them per tree, fit fresh on each fold's training rows inside the CV loop below) -- so, unlike Stage 2's Ridge models, Stage 1 needs no separate imputer step and therefore no global `.fit_transform()`/`.fit()` call on the full sample set at all, sidestepping that leakage path entirely rather than making it fold-safe.

In [ ]:
X_all = data_merged[spc_feat_cols + conf_feat_cols].values.astype(float)

lgb_clf = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    n_jobs=2,
    verbose=-1,
)

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_proba = np.zeros((len(y_class), n_classes))
oof_preds = np.zeros(len(y_class), dtype=int)
fold_accs = []

for fold, (tr_idx, val_idx) in enumerate(cv10.split(X_all, y_class)):
    lgb_clf.fit(X_all[tr_idx], y_class[tr_idx])
    oof_proba[val_idx] = lgb_clf.predict_proba(X_all[val_idx])
    oof_preds[val_idx] = lgb_clf.predict(X_all[val_idx])
    fold_accs.append((oof_preds[val_idx] == y_class[val_idx]).mean())

print(f"Stage 1 OOF accuracy: {np.mean(fold_accs):.3f} +/- {np.std(fold_accs):.3f}")
print(f"Cohen Kappa: {cohen_kappa_score(y_class, oof_preds):.3f}")
print("\nClassification report:")
print(classification_report(y_class, oof_preds, target_names=STAGE_GROUP_ORDER))

proba_cols = [f'P_{g}' for g in STAGE_GROUP_ORDER]
proba_df = pd.DataFrame(oof_proba, columns=proba_cols)
proba_df['Sample'] = samples


## 3. Stage 2 -- Ridge regression per polyamine (two-stage vs flat, leakage-safe pipeline)

Target polyamines are NB03's **pre-specified** polyamine targets (`ml_results_v2.pkl`,
key `polyamine_targets`) -- the study's a-priori hypothesis metabolites, analysed whether
or not they are FDR-dysregulated. (The FDR-dysregulated subset is printed for reference.)

In [ ]:
dysreg_pa = pd.concat(
    [ml_res['dysreg_polyamines_early'], ml_res['dysreg_polyamines_advanced']],
    ignore_index=True,
).drop_duplicates(subset='Feature')
print(f"FDR-dysregulated polyamines (reference only): {dysreg_pa['Feature'].tolist()}")

pa_targets = ml_res.get('polyamine_targets', pd.DataFrame())
target_pa_cols = [c for c in pa_targets.get('Feature', pd.Series(dtype=str)).tolist() if c in mtb.columns]
if not target_pa_cols:
    target_pa_cols = [v for v in pa_cols.values() if v in mtb.columns]

print(f"Target polyamines for Stage 2 ({len(target_pa_cols)}): {target_pa_cols}")

# Sample-key merge (not positional concatenation) -- proba_df was built
# directly from data_merged's own row order so this merge cannot drop or
# reorder anything, but building it this way means every downstream
# per-polyamine matrix is also assembled by key, so a polyamine with fewer
# non-missing samples than the full cohort can never silently misalign X
# against y.
base_df = data_merged[['Sample'] + spc_feat_cols + conf_feat_cols].merge(proba_df, on='Sample')
assert len(base_df) == len(data_merged), "proba_df merge must not drop/duplicate rows"


def build_pa_xy(pa_col):
    """Row-aligned two-stage/flat feature matrices + target for one polyamine.

    Rows with a missing target value are dropped outright rather than
    imputed -- the old notebook's fold-unsafe global median fill is the
    second leakage bug this notebook fixes; dropping instead of imputing
    is leakage-proof by construction (a discarded row cannot leak
    information into any fold).
    """
    pa_df = base_df.merge(mtb[['Sample', pa_col]], on='Sample').dropna(subset=[pa_col])
    X_ts   = pa_df[spc_feat_cols + proba_cols + conf_feat_cols].values.astype(float)
    X_flat = pa_df[spc_feat_cols + conf_feat_cols].values.astype(float)
    y      = pa_df[pa_col].values.astype(float)
    return X_ts, X_flat, y, len(pa_df)


def make_ridge_pipeline():
    """Fresh Pipeline per model: IterativeImputer -> StandardScaler -> Ridge.

    cross_val_predict clones this pipeline once per CV split and fits the
    clone on that split's training rows only, so no imputer/scaler
    statistic is ever computed from a held-out sample -- this replaces the
    old notebook's scaler.fit_transform(X) call made once on the full
    sample set before cross_val_predict.
    """
    # skip_complete=True: only the few confounder columns with NaN are modelled
    # (otherwise IterativeImputer fits a regressor for all ~500 complete columns).
    return Pipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=42, skip_complete=True)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0)),
    ])


In [ ]:
results_stage2 = []
cv10_reg = KFold(n_splits=10, shuffle=True, random_state=42)

for pa_col in target_pa_cols:
    X_twostage, X_flat, y_pa, n_used = build_pa_xy(pa_col)
    n_dropped = len(base_df) - n_used
    if n_dropped:
        print(f"  {pa_col}: {n_dropped} sample(s) dropped (missing target / not in mtb)")
    if len(y_pa) < 20:
        print(f"  {pa_col}: only {len(y_pa)} usable samples -- skipping")
        continue

    pa_kegg = {v: k for k, v in pa_cols.items()}.get(pa_col, pa_col)
    pa_name = POLYAMINE_KEGG.get(pa_kegg, pa_kegg)

    pipe_ts = make_ridge_pipeline()
    y_pred_ts = cross_val_predict(pipe_ts, X_twostage, y_pa, cv=cv10_reg)
    r_ts = pearsonr(y_pa, y_pred_ts)[0]

    pipe_flat = make_ridge_pipeline()
    y_pred_flat = cross_val_predict(pipe_flat, X_flat, y_pa, cv=cv10_reg)
    r_flat = pearsonr(y_pa, y_pred_flat)[0]

    results_stage2.append({
        'Polyamine':   pa_name,
        'Feature':     pa_col,
        'N':           n_used,
        'R_two_stage': round(r_ts, 3),
        'R_flat':      round(r_flat, 3),
        'Delta_R':     round(r_ts - r_flat, 3),
    })
    print(f"  {pa_name}: two-stage R={r_ts:.3f}, flat R={r_flat:.3f}, Delta_R={r_ts - r_flat:+.3f}")

results_df = pd.DataFrame(results_stage2)
print(results_df.to_string(index=False))

out_csv = RESULTS_DIR / 'tables' / 'two_stage_model_results_v2.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")


## 4. SHAP attribution -- every target polyamine

The old notebook computed SHAP inside the main loop for every polyamine but then only ever ran this explanation step on `target_pa_cols[0]`, an undocumented restriction to a single metabolite. Here the loop below runs for **every** entry in `target_pa_cols`, saving one SHAP table and one figure each. Each polyamine's final model is refit on all of that polyamine's own valid rows (via the same leakage-safe `Pipeline`) purely for attribution -- the R_two_stage/R_flat performance numbers above come exclusively from the per-fold `cross_val_predict` pipelines, never from this full-data refit.

In [ ]:
def _pa_slug(name: str) -> str:
    """Filesystem-safe stub for filenames (POLYAMINE_KEGG names are clean,
    but the KEGG-id/raw-column fallback can contain parens/commas)."""
    return re.sub(r'[^a-z0-9]+', '_', str(name).lower()).strip('_')


tables_dir = RESULTS_DIR / 'tables'
figs_dir   = RESULTS_DIR / 'figures' / 'ml'
tables_dir.mkdir(parents=True, exist_ok=True)
figs_dir.mkdir(parents=True, exist_ok=True)

feat_names_ts = spc_feat_cols + proba_cols + conf_feat_cols
shap_summary_rows = []

for pa_col in target_pa_cols:
    X_twostage, _X_flat, y_pa, n_used = build_pa_xy(pa_col)
    if len(y_pa) < 20:
        continue

    pa_kegg = {v: k for k, v in pa_cols.items()}.get(pa_col, pa_col)
    pa_name = POLYAMINE_KEGG.get(pa_kegg, pa_kegg)
    slug = _pa_slug(pa_name)

    pipe_final = make_ridge_pipeline()
    pipe_final.fit(X_twostage, y_pa)

    # Transform through imputer+scaler only, to get the exact input space
    # the fitted Ridge coefficients operate on.
    X_pre = pipe_final[:-1].transform(X_twostage)
    ridge_final = pipe_final.named_steps['ridge']

    explainer = shap.LinearExplainer(ridge_final, X_pre)
    shap_values = explainer.shap_values(X_pre)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    shap_df = pd.DataFrame({'Feature': feat_names_ts, 'Mean_SHAP': mean_abs_shap})
    shap_df = shap_df.sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

    top20 = shap_df.head(20)
    colors = ['#e74c3c' if f in proba_cols else '#3498db' if f in conf_feat_cols else '#2ecc71'
              for f in top20['Feature']]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top20['Feature'][::-1], top20['Mean_SHAP'][::-1], color=colors[::-1])
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'Stage 2 SHAP -- {pa_name} (two-stage model, n={n_used})\n'
                 '(red=stage proba, blue=confounder, green=species)')
    plt.tight_layout()

    out_fig = figs_dir / f'two_stage_shap_{slug}_v2.png'
    fig.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    out_shap_csv = tables_dir / f'two_stage_shap_{slug}_v2.csv'
    shap_df.to_csv(out_shap_csv, index=False)

    shap_summary_rows.append({
        'Polyamine':     pa_name,
        'Feature':       pa_col,
        'N':             n_used,
        'Top_feature':   shap_df.iloc[0]['Feature'],
        'Top_Mean_SHAP': round(float(shap_df.iloc[0]['Mean_SHAP']), 4),
    })
    print(f"  {pa_name}: saved {out_shap_csv.name}, {out_fig.name}")

shap_summary_df = pd.DataFrame(shap_summary_rows)
print(f"\nSHAP computed for {len(shap_summary_df)}/{len(target_pa_cols)} target polyamine(s)")
print(shap_summary_df.to_string(index=False))


## Summary

In [ ]:
print("=== Two-Stage Model Summary (v2) ===")
print(f"\nStage 1 (LightGBM {n_classes}-class: {STAGE_GROUP_ORDER}):")
print(f"  OOF accuracy: {np.mean(fold_accs):.3f} +/- {np.std(fold_accs):.3f}")
print(f"  Cohen Kappa:  {cohen_kappa_score(y_class, oof_preds):.3f}")

print(f"\nStage 2 comparison (10-fold CV Pearson R, leakage-safe pipeline):")
if not results_df.empty:
    print(results_df[['Polyamine', 'N', 'R_two_stage', 'R_flat', 'Delta_R']].to_string(index=False))
else:
    print("  (no target polyamines produced usable results)")

print(f"\nSHAP explanation computed for {len(shap_summary_df)}/{len(target_pa_cols)} "
      f"target polyamine(s): {', '.join(target_pa_cols)}")

print("\nOutputs:")
print("  - tables/two_stage_model_results_v2.csv")
print("  - tables/two_stage_shap_<polyamine>_v2.csv        (one per target polyamine)")
print("  - figures/ml/two_stage_shap_<polyamine>_v2.png    (one per target polyamine)")

print("\nThis notebook is a self-contained leaf: no downstream v2 notebook reads "
      "two_stage_model_results_v2.csv or the SHAP tables/figures it writes.")
